# Route labels — the golden set over the cell composition

**Input.** `CellFill().build()` → `data/composition/cell_selection.parquet`: the
query set the 44 archetype cells (`cells.yaml`) selected, ~51.5K distinct
queries across 42 datasets. That is the dataset; nothing here invents queries.

**Output.** `data/route_labels/labels.parquet`: **one row per labelled
`(dataset, query_id)`** carrying the *route label* — which of `dense_only`,
`pure_rrf`, `sparse_only` retrieved best — the losing routes' scores, the
outcome shape, and the selection's own `cell` / `checkable` columns. A query
that fills several cells is labelled once; per-cell readouts join back to
`cell_selection` on `query_id`.

**How the label is decided.** All three routes are run, each ranking scored by
the router objective, argmax wins. No LLM is asked which route is better:
dense-vs-sparse depends on the corpus vocabulary and its IDF, neither of which
is in the query, so a model reading only the query is being asked to predict
something that isn't a function of its input.

**Coverage.** Every one of the 42 datasets has its corpus materialized on disk
(this session's `materialize_corpora.py` run), so the only gap between selected
and labelled is Qdrant indexing — closed by the size-ascending sweep in §17.
The anchor sections (§4–§15) walk two datasets in detail first; the sweep then
labels the rest. `min_relevance` is per-lane from `LANES` (antique 3, trec-dl 2,
the rest 1), so graded lanes are binarized at the grade their benchmark intends.

**Prerequisites**

```bash
docker compose up -d          # local Qdrant on :6333
```

In [3]:
%load_ext autoreload
%autoreload 2

## 1 — Setup

In [9]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from qdrant_client import QdrantClient
from qdrant_client.models import Distance

load_dotenv(".env") or load_dotenv("../.env")

DENSE_MODEL, DENSE_SIZE = "BAAI/bge-small-en-v1.5", 384
SPARSE_MODEL = "Qdrant/bm25"

client = QdrantClient(
    url=os.getenv("QDRANT_URL", "http://localhost:6333"),
    api_key=os.getenv("QDRANT_API_KEY"),
)


def show(df: pd.DataFrame) -> None:
    display(Markdown(df.to_markdown(index=False)))


print("qdrant collections:", [c.name for c in client.get_collections().collections])

qdrant collections: ['antique_routes', 'bright-aops_routes', 'bright-biology_routes', 'bright-earth-science_routes', 'bright-economics_routes', 'bright-leetcode_routes', 'bright-pony_routes', 'bright-psychology_routes', 'bright-robotics_routes', 'bright-stackoverflow_routes', 'bright-sustainable-living_routes', 'bright-theoremqa-questions_routes', 'bright-theoremqa-theorems_routes', 'clerc_routes', 'clerc_v5', 'composition_selection', 'crumb-clinical-trial_routes', 'crumb-code-retrieval_routes', 'crumb-legal-qa_routes', 'crumb-paper-retrieval_routes', 'crumb-set-operation-entity-retrieval_routes', 'crumb-stack-exchange_routes', 'crumb-theorem-retrieval_routes', 'crumb-tip-of-the-tongue_routes', 'dbpedia-entity_routes', 'freshstack-angular_routes', 'freshstack-godot_routes', 'freshstack-langchain_routes', 'freshstack-laravel_routes', 'freshstack-yolo_routes', 'gooaq_routes', 'limit_routes', 'lotte-technology-forum_routes', 'lotte-technology-search_routes', 'miracl-en-dev_routes', 'msmar

## 2 — Load the composition

`CellFill().build()` returns the cell selection, reading
`cell_selection.parquet` from disk when it exists — so this does not re-run the
fill. `RouteLabels` is selection-agnostic: it carries whichever of
`cell` / `stage` / `route` / `checkable` the frame has.

In [10]:
from composition import CellFill
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = CellFill().build()   # data/composition/cell_selection.parquet — the 44-cell query set
print(f"selection: {len(selection):,} rows x {len(selection.columns)} cols "
      f"({selection[['dataset','query_id']].drop_duplicates().shape[0]:,} distinct queries)")
show(selection.sample(5))

labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
print("objective:", labels.objective.name)

selection: 8,471 rows x 9 cols (7,099 distinct queries)


| dataset             | query_id                                       | checkable   | cell                         | stage     | route       | provenance   | floors                                                        | query                                                                                                                                                                |
|:--------------------|:-----------------------------------------------|:------------|:-----------------------------|:----------|:------------|:-------------|:--------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| orcas               | 6557627                                        | True        |                              | control   | nan         | natural      | []                                                            | icd cardiac device                                                                                                                                                   |
| scirgen-geo-en      | 023474e5-57b9-4132-9e9c-0f3d57138bd0-Example-1 | True        | multi_statement_context_dump | reused    | dense_only  | natural      | ['verbose_grammatical_request' 'multi_statement_context_dump' | Could you provide an example of a dataset that could be used to verify the accuracy of tree height extraction methods?                                               |
|                     |                                                |             |                              |           |             |              |  'conversational_courtesy_wrapper']                           |                                                                                                                                                                      |
| msmarco-passage-dev | 1093856                                        | True        | stopword_saturated_midlength | reused    | sparse_only | natural      | ['stopword_saturated_midlength']                              | is venus a gas planet or a terrestrial                                                                                                                               |
| webfaq-eng          | 4847709                                        | True        | verbose_grammatical_request  | candidate | nan         | natural      | ['verbose_grammatical_request']                               | Why is it an important consideration in promoting a national brand name to an international audience whether or not to keep the brand name in its original language? |
| gooaq               | 3214037                                        | True        |                              | control   | nan         | natural      | []                                                            | what is gta v next gen?                                                                                                                                              |

objective: 0.7*HitRate@1+0.3*NDCG@10


## 3 — What can be labelled today

`unlabelled` means the row is in the composition but its dataset has no
indexed corpus or no qrels on disk. This table is the real progress bar for
the golden set, and it should be re-read after every dataset lands.

In [11]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,} "
      f"({coverage.unlabelled.sum() / coverage.selected.sum() * 100:.1f}%)")

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| orcas                                |       1491 |        510 |             0 |          981 |             499 |         10 |          1 |
| webfaq-eng                           |       1040 |        490 |             0 |          550 |             475 |         14 |          1 |
| scirgen-geo-en                       |        847 |        471 |           230 |          376 |             454 |          8 |          9 |
| rarb-math                            |        733 |        462 |            75 |          271 |             462 |          0 |          0 |
| gooaq                                |        774 |        390 |             0 |          384 |             381 |          9 |          0 |
| msmarco-passage-dev                  |       1286 |        329 |             0 |          957 |             320 |          9 |          0 |
| clerc                                |        327 |        284 |             0 |           43 |             284 |          0 |          0 |
| crumb-legal-qa                       |        374 |        234 |            78 |          140 |             234 |          0 |          0 |
| crumb-code-retrieval                 |        220 |        177 |             0 |           43 |             177 |          0 |          0 |
| lotte-technology-forum               |        139 |        126 |             0 |           13 |             126 |          0 |          0 |
| quest                                |        125 |        116 |             0 |            9 |             115 |          1 |          0 |
| rarb-code                            |        123 |        113 |             0 |           10 |             113 |          0 |          0 |
| freshstack-laravel                   |         67 |         67 |             0 |            0 |              67 |          0 |          0 |
| freshstack-langchain                 |         66 |         66 |             0 |            0 |              66 |          0 |          0 |
| freshstack-angular                   |         52 |         52 |             0 |            0 |              52 |          0 |          0 |
| bright-earth-science                 |         50 |         50 |             0 |            0 |              50 |          0 |          0 |
| limit                                |         54 |         49 |             5 |            5 |              49 |          0 |          0 |
| beir-nfcorpus                        |         38 |         38 |             0 |            0 |               2 |          0 |         36 |
| lotte-technology-search              |         40 |         37 |             0 |            3 |              37 |          0 |          0 |
| crumb-stack-exchange                 |         39 |         37 |             0 |            2 |              37 |          0 |          0 |
| bright-biology                       |         34 |         34 |             0 |            0 |              34 |          0 |          0 |
| crumb-clinical-trial                 |         31 |         29 |             0 |            2 |              29 |          0 |          0 |
| bright-pony                          |         28 |         28 |             0 |            0 |              28 |          0 |          0 |
| bright-sustainable-living            |         27 |         27 |             0 |            0 |              27 |          0 |          0 |
| antique                              |         24 |         23 |             0 |            1 |              23 |          0 |          0 |
| crumb-set-operation-entity-retrieval |         33 |         20 |            10 |           13 |              20 |          0 |          0 |
| bright-psychology                    |         19 |         19 |             0 |            0 |              19 |          0 |          0 |
| bright-economics                     |         18 |         18 |             0 |            0 |              18 |          0 |          0 |
| freshstack-godot                     |         18 |         18 |             0 |            0 |              18 |          0 |          0 |
| freshstack-yolo                      |         18 |         18 |             0 |            0 |              18 |          0 |          0 |
| bright-leetcode                      |         18 |         16 |             0 |            2 |              16 |          0 |          0 |
| dbpedia-entity                       |         15 |         15 |             0 |            0 |              15 |          0 |          0 |
| bright-robotics                      |         15 |         15 |             0 |            0 |              15 |          0 |          0 |
| bright-theoremqa-questions           |         16 |         14 |             0 |            2 |              14 |          0 |          0 |
| trec-dl-2022                         |        208 |         13 |             0 |          195 |              13 |          0 |          0 |
| miracl-en-dev                        |         20 |         13 |             5 |            7 |              13 |          0 |          0 |
| bright-theoremqa-theorems            |         12 |         12 |             0 |            0 |              12 |          0 |          0 |
| bright-stackoverflow                 |         11 |         11 |             0 |            0 |              11 |          0 |          0 |
| crumb-paper-retrieval                |         11 |          9 |             0 |            2 |               9 |          0 |          0 |
| crumb-theorem-retrieval              |          5 |          4 |             0 |            1 |               4 |          0 |          0 |
| bright-aops                          |          4 |          2 |             1 |            2 |               2 |          0 |          0 |
| crumb-tip-of-the-tongue              |          1 |          0 |             1 |            1 |               0 |          0 |          0 |

selected 8,471 | labelled 4,456 | unlabelled 4,015 (47.4%)


## 4 — Index one dataset's corpus

Starting with `beir-nfcorpus`: 3,633 documents, small enough to index in full
with no corpus sampling, and it ships human judgments so the labels are
checkable. It is also richly judged — 38.2 judged docs per query, and 300 of its
323 queries have two or more relevant documents, which is the only regime where
the choice of objective can change a label at all.

**Not** reusing the existing `nf` collection: it holds 33,633 points — 3,633
nfcorpus documents mixed with 30,000 trec-dl passages — so nfcorpus queries
would be scored against a corpus that is 90% unrelated.

Dense and sparse live as two named vector slots on one collection, which is what
lets a single Qdrant call fuse them. Idempotent: re-running skips the upload.

In [12]:
from hybrid_search_rrf_dataset.indexer import (
    CorpusDocument,
    CorpusIndexer,
    EmbeddingCache,
    EmbeddingConfig,
)
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset

DATASET = "beir-nfcorpus"        # the composition's key
COLLECTION = "nfcorpus_routes"

# snapshot dir is the lane's source.name (beir-nfcorpus), not the upstream slug
source = SnapshotDataset(DATASET, path="data")   # written by RetrievalDataset.save()
corpus = source.corpus()

dense_cfg = EmbeddingConfig(
    name="dense_base", model_id=DENSE_MODEL, kind="dense",
    size=DENSE_SIZE, distance=Distance.COSINE,
    parallel=4
)
sparse_cfg = EmbeddingConfig(name="sparse_base", model_id=SPARSE_MODEL, kind="sparse")

indexer = CorpusIndexer(
    client, COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
indexer.ensure_collection()

if client.count(COLLECTION, exact=True).count >= len(corpus):
    print(f"{COLLECTION}: already indexed — skipping upload")
else:
    indexer.upload([CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64)
print(f"{COLLECTION}: {client.count(COLLECTION, exact=True).count:,} points")

FileNotFoundError: data/beir-nfcorpus is missing ['queries.parquet', 'qrels.parquet'].

## 5 — The three routes

`DenseOnlyStrategy` and `SparseOnlyStrategy` keep their raw scores;
`PureRRFStrategy` is Qdrant-native RRF, matching production's `Fusion::Rrf`.

RRF fuses by *rank position*, which is exactly why it can lose on top-1: a doc
ranked first by dense and 40th by sparse loses to one ranked third by both.

In [13]:
from hybrid_search_rrf_dataset.fusion import (
    DenseOnlyStrategy,
    PureRRFStrategy,
    SparseOnlyStrategy,
)

args = (client, COLLECTION, dense_cfg, sparse_cfg)
dense, hybrid, sparse = (
    DenseOnlyStrategy(*args), PureRRFStrategy(*args), SparseOnlyStrategy(*args)
)

probe = labels.rows_for(DATASET)["query"].iloc[0]
print(f"probe (a real selection row): {probe!r}\n")
for s in (dense, hybrid, sparse):
    top = list(s.rank(probe).items())[:3]
    print(f"  {s.name:12s} " + "  ".join(f"{d}={v:.3f}" for d, v in top))

NameError: name 'dense_cfg' is not defined

## 6 — Label this dataset's selection rows

`RouteLabels.label` narrows the source to the composition's query ids via
`QuerySubset`, runs `GoldenRoutingBuilder`, and merges the result into
`labels.parquet` — replacing only this dataset's rows.

The objective is `0.7·HitRate@1 + 0.3·NDCG@10`. Because the hit weight exceeds
the NDCG weight the score ranges are disjoint (rank-1 hit ⇒ ≥0.700, miss ⇒
≤0.300), so it is lexicographic: top-1 decides, NDCG@10 only breaks ties within
each group. `min_relevance=1` because nfcorpus grades are 1 and 2 only.

In [22]:
labelled = labels.label(source, dense, hybrid, sparse, dataset=DATASET)
print(f"labelled {len(labelled):,} rows -> {labels.labels_path}")
show(labelled.head(8).round(3))

labelled 12 rows -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet


| dataset       | query_id   | route       | query                                                        |   score |   score_dense_only |   score_pure_rrf |   score_sparse_only | shape         | metric_name               |   min_relevance | checkable   | cell                         | stage   | route_selected   |
|:--------------|:-----------|:------------|:-------------------------------------------------------------|--------:|-------------------:|-----------------:|--------------------:|:--------------|:--------------------------|----------------:|:------------|:-----------------------------|:--------|:-----------------|
| beir-nfcorpus | PLAIN-23   | sparse_only | How to Reduce Exposure to Alkylphenols Through Your Diet     |   0.921 |              0.847 |            0.914 |               0.921 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | stopword_saturated_midlength | reused  | dense_only       |
| beir-nfcorpus | PLAIN-383  | sparse_only | What do you think of Dr. Jenkins' take on paleolithic diets? |   0.051 |              0.044 |            0.051 |               0.051 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | stopword_saturated_midlength | reused  | sparse_only      |
| beir-nfcorpus | PLAIN-1183 | sparse_only | Finland                                                      |   0.787 |              0.026 |            0.061 |               0.787 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | bare_concept_token           | reused  | sparse_only      |
| beir-nfcorpus | PLAIN-1409 | dense_only  | industrial toxins                                            |   0.872 |              0.872 |            0.091 |               0.026 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | bare_concept_token           | reused  | dense_only       |
| beir-nfcorpus | PLAIN-1463 | sparse_only | kidney beans                                                 |   0.839 |              0.064 |            0.118 |               0.839 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | bare_concept_token           | reused  | sparse_only      |
| beir-nfcorpus | PLAIN-1537 | dense_only  | low-carb diets                                               |   0.884 |              0.884 |            0.12  |               0.028 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | keyword_telegram_short       | reused  | dense_only       |
| beir-nfcorpus | PLAIN-1721 | sparse_only | NIH-AARP study                                               |   0.899 |              0     |            0.094 |               0.899 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | keyword_telegram_short       | reused  | sparse_only      |
| beir-nfcorpus | PLAIN-1950 | dense_only  | prunes                                                       |   0.799 |              0.799 |            0.075 |               0.075 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | bare_concept_token           | reused  | dense_only       |

## 7 — Outcome shapes

Only one of the three shapes teaches the router about dense-vs-sparse.

| shape | meaning |
| --- | --- |
| `routes_differ` | the quality signal — the trainable set |
| `all_tied` | any route works; serve the cheapest. Signal for the *speed* goal |
| `all_zero` | nothing relevant found by any route — unanswerable, and **no valid label exists**. `route` is stored null |

Read per **cell** — the archetype the query was selected into. The carried
`cell` is one representative when a query fills several; the exact per-cell
view joins `labels` back to `cell_selection` on `query_id` so a multi-cell
query counts under each of its cells. The empty `cell` is the control /
feature-blind draw.

In [23]:
shapes = (labelled.groupby(["cell", "shape"]).size()
          .unstack(fill_value=0))
shapes["total"] = shapes.sum(axis=1)
show(shapes.reset_index())

overall = labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = labelled[labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

| cell                         |   routes_differ |   total |
|:-----------------------------|----------------:|--------:|
| bare_concept_token           |               4 |       4 |
| high_morphological_variation |               1 |       1 |
| keyword_telegram_short       |               3 |       3 |
| short_grammatical_question   |               1 |       1 |
| stopword_saturated_midlength |               3 |       3 |

| shape         |   rows | share   |
|:--------------|-------:|:--------|
| routes_differ |     12 | 100.0%  |

route distribution over the 12 trainable rows:


| route       |   rows | share   |
|:------------|-------:|:--------|
| sparse_only |      6 | 50.0%   |
| dense_only  |      6 | 50.0%   |

majority-class baseline: 50% (always predict sparse_only)


## 8 — Constant-route baselines

The bar a router must clear is **the best constant route**, not random. A
candidate can post respectable regret against the oracle and still lose to one
line of code, so these belong in every comparison.

In [8]:
from hybrid_search_rrf_dataset.evaluation import compare
from hybrid_search_rrf_dataset.golden import BaselineBuilder, GoldenRoutingBuilder
from hybrid_search_rrf_dataset.retrieval import QuerySubset

subset = QuerySubset(source, labels.rows_for(DATASET)["query_id"])
oracle = GoldenRoutingBuilder(
    dense, hybrid, sparse, objective=labels.objective
).build_or_load(subset, path=f"data/route_labels/{DATASET}_oracle")

rows = []
for strategy in (dense, hybrid, sparse):
    const = BaselineBuilder(strategy, objective=labels.objective).build_or_load(
        subset, path=f"data/route_labels/{DATASET}_const_{strategy.name}"
    )
    s = compare(oracle, const)
    rows.append({
        "always pick": str(strategy.name),
        "mean score": round(s.mean_metric_candidate, 3),
        "oracle ceiling": round(s.mean_metric_golden, 3),
        "mean regret": round(s.mean_regret, 3),
        "p90 regret": round(s.p90_regret, 3),
        "reaches oracle": f"{s.oracle_hit_rate_pct:.0f}%",
        "route agreement": f"{s.route_agreement_pct:.0f}%",
    })
show(pd.DataFrame(rows))

NameError: name 'source' is not defined

## 9 — Objective sensitivity, at zero retrieval cost — pooled across every lane

`route_rankings` holds each route's top-10 doc ids and the judgments live in
`QrelStore`, so any metric depending only on the *order* of the top 10 can be
recomputed without touching Qdrant. That is what makes the objective a
reversible decision instead of a one-way door.

Originally this cell recomputed for the single nfcorpus anchor dataset only.
It now pools across **every lane with a cached oracle** —
`data/route_labels/{lane}_oracle/rows.parquet`, one retrieval pass per lane,
built once via `GoldenRoutingBuilder.build_or_load`. Each variant threads the
*lane's own* `min_relevance` through (antique binarizes at grade 3, trec-dl at
2, the rest at 1) rather than a single flat value — the "shipped" column must
reproduce exactly what `labels.parquet` actually stores, or the flip-rate
comparison is comparing against a fiction.

We synthesize descending scores from the stored order — HitRate@1 and NDCG@10
depend only on that order, so the recomputation is exact. It cannot evaluate a
metric needing rank 11+ or the raw retrieval scores.

Safe to run before every lane finishes caching: an uncached lane is skipped
and counted, not a crash — re-run later for fuller coverage.

In [6]:
from pathlib import Path

from hybrid_search_rrf_dataset.fusion import derive_route
from hybrid_search_rrf_dataset.golden import GoldenRoutingBuilder
from hybrid_search_rrf_dataset.lanes import LANES
from hybrid_search_rrf_dataset.objective import NDCGObjective
from hybrid_search_rrf_dataset.qrels import QrelStore
from hybrid_search_rrf_dataset.retrieval import SnapshotDataset


def load_lane_oracle(key: str):
    path = Path("data/route_labels") / f"{key}_oracle"
    if not (path / "rows.parquet").exists():
        return None
    return GoldenRoutingBuilder.load(path=path)


lane_rows, lane_lookup, skipped = {}, {}, []
for key in LANES:
    rows = load_lane_oracle(key)
    if rows is None:
        skipped.append(key)
        print(f"{key:40s} skip — not cached")
        continue
    lane_source = SnapshotDataset(LANES[key].source.name, path="data")
    lane_rows[key] = rows
    lane_lookup[key] = QrelStore.from_dataset(lane_source).lookup(lane_source.name)
    print(f"{key:40s} loaded {len(rows):>6,} rows")


all_rows = [(key, row) for key, rows in lane_rows.items() for row in rows]
print(f"pooled oracle rows: {len(all_rows):,} across {len(lane_rows)} of {len(LANES)} lanes")
if skipped:
    print(f"not yet cached ({len(skipped)}): {', '.join(skipped)}")

beir-nfcorpus                            skip — not cached
msmarco-passage-dev                      skip — not cached
trec-dl-2022                             skip — not cached
rarb-math                                skip — not cached
rarb-code                                skip — not cached
bright-aops                              skip — not cached
bright-leetcode                          skip — not cached
bright-theoremqa-questions               skip — not cached
bright-biology                           skip — not cached
bright-earth-science                     skip — not cached
bright-economics                         skip — not cached
bright-pony                              skip — not cached
bright-psychology                        skip — not cached
bright-robotics                          skip — not cached
bright-stackoverflow                     skip — not cached
bright-sustainable-living                skip — not cached
bright-theoremqa-theorems                skip — not cach

In [7]:
from tqdm.auto import tqdm


def relabel(make_objective, desc: str) -> pd.Series:
    # derive_route, not argmax: ties resolve to the cheapest route (SPEC d41),
    # matching exactly how labels.parquet's `route` column is derived — a plain
    # max() here would silently default every tie to dense_only (StrategyName's
    # enum order), corrupting the flip-rate comparison against `shipped`.
    picks = []
    for key, row in tqdm(all_rows, desc=desc):
        objective = make_objective(LANES[key].min_relevance)
        gold = lane_lookup[key].get(row.query_id, {})
        scored = {
            route: objective.assess(
                {d: 1.0 / (i + 1) for i, d in enumerate(ids)}, gold
            )[0]
            for route, ids in row.route_rankings.items()
        }
        picks.append(str(derive_route(scored)))
    return pd.Series(picks)


shipped = pd.Series([str(row.strategy_name) for _, row in all_rows])
variants = {
    "1·HR@1": lambda mr: RouterObjective(hit_weight=1, ndcg_weight=0, min_relevance=mr),
    "0.7·HR@1 + 0.3·NDCG@10  (shipped)": lambda mr: RouterObjective(min_relevance=mr),
    "0.5·HR@1 + 0.5·NDCG@10": lambda mr: RouterObjective(hit_weight=0.5, ndcg_weight=0.5, min_relevance=mr),
    "bare NDCG@10 (no top-1 term)": lambda mr: NDCGObjective(min_relevance=mr),
    "shipped, stricter min_relevance+1": lambda mr: RouterObjective(min_relevance=mr + 1),
}

rows_out = []
for name, make_objective in variants.items():
    picks = relabel(make_objective, desc=name)
    result = {
        "objective": name,
        "flips": int((picks != shipped).sum()),
        "flip rate": f"{(picks != shipped).mean() * 100:.1f}%",
        **{f"picks {r}": int((picks == r).sum())
           for r in ("dense_only", "pure_rrf", "sparse_only")},
    }
    print(f"  -> {result['flips']:,} flips ({result['flip rate']})")
    rows_out.append(result)
show(pd.DataFrame(rows_out))

1·HR@1: 0it [00:00, ?it/s]


  -> 0 flips (nan%)


0.7·HR@1 + 0.3·NDCG@10  (shipped): 0it [00:00, ?it/s]


  -> 0 flips (nan%)


0.5·HR@1 + 0.5·NDCG@10: 0it [00:00, ?it/s]


...

bare NDCG@10 (no top-1 term): 0it [00:00, ?it/s]


  -> 0 flips (nan%)


shipped, stricter min_relevance+1: 0it [00:00, ?it/s]

  -> 0 flips (nan%)


| objective                         |   flips | flip rate   |   picks dense_only |   picks pure_rrf |   picks sparse_only |
|:----------------------------------|--------:|:------------|-------------------:|-----------------:|--------------------:|
| 1·HR@1                            |       0 | nan%        |                  0 |                0 |                   0 |
| 0.7·HR@1 + 0.3·NDCG@10  (shipped) |       0 | nan%        |                  0 |                0 |                   0 |
| 0.5·HR@1 + 0.5·NDCG@10            |       0 | nan%        |                  0 |                0 |                   0 |
| bare NDCG@10 (no top-1 term)      |       0 | nan%        |                  0 |                0 |                   0 |
| shipped, stricter min_relevance+1 |       0 | nan%        |                  0 |                0 |                   0 |

## 10 — Where the golden set stands

Re-read coverage now that one dataset is labelled, and project the trainable
yield. The projection assumes other datasets behave like this one, which they
will not — nfcorpus is a single 3,633-document medical corpus, unusually
richly judged. Treat it as an order of magnitude, not a forecast.

In [20]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])

done = labels.load()
trainable = (done["shape"] == "routes_differ").mean()
print(f"labelled so far:        {len(done):,} of {len(selection):,} rows")
print(f"trainable share:        {trainable * 100:.1f}%")
print(f"projected yield at 50K: ~{int(trainable * len(selection)):,} rows "
      f"(extrapolated from one dataset — see caveat above)")
print()
print("next unblock, largest first:")
show(coverage[coverage.labelled == 0].head(5)[["dataset", "selected", "unlabelled"]])

| dataset       |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:--------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| beir-nfcorpus |         12 |         12 |             0 |            0 |              12 |          0 |          0 |

labelled so far:        12 of 57,653 rows
trainable share:        100.0%
projected yield at 50K: ~57,653 rows (extrapolated from one dataset — see caveat above)

next unblock, largest first:


| dataset             |   selected |   unlabelled |
|:--------------------|-----------:|-------------:|
| webfaq-eng          |       9345 |         9345 |
| msmarco-passage-dev |       9245 |         9245 |
| scirgen-geo-en      |       8288 |         8288 |
| orcas               |       7782 |         7782 |
| gooaq               |       7296 |         7296 |

## 11 — msmarco-passage-dev: the composition's largest lane (SPEC d38)

15,678 composition rows — 31% of the 50K. The local dev qrels cover 7,697 of
them (49.1%); the rest stay `unlabelled` in coverage — a gap to report, never
a licence to substitute other queries. The gap is MS MARCO's own: the full
dev set ships 101,093 queries but judgments were released for only 55,578
(55%), and the fill drew judgment-blind — the composition's `checkable=True`
was assigned per-dataset, not per-query. Median **one** judged passage per
query against nfcorpus's 16, so this lane sits at the opposite end of the
judgment-density axis: the regime where two different top-10 lists cannot
both be right.

The corpus recipe (d38c): every judged-relevant passage for the selected
queries is force-included, then padded with uniform-random passages from the
full 8.8M collection to **100,000** total, fixed seed. Uniform sampling
preserves the collection's vocabulary/IDF profile — the d37(g) fix: trec-dl's
judged-docs-only corpus was near-all answers, which inflates dense and
starves sparse.

Materialization is one-time and local (the ir_datasets collection is already
on disk; first `docs_store` access builds its index). Re-runs read the
snapshot back via `SnapshotDataset`.

In [11]:
from pathlib import Path

from hybrid_search_rrf_dataset.retrieval import MSMarcoDev

MS_DATASET = "msmarco-passage-dev"      # composition key == source name here
MS_COLLECTION = "msmarco_routes"

if not (Path("data") / MS_DATASET / "corpus.parquet").exists():
    ms = MSMarcoDev(
        query_ids=labels.rows_for(MS_DATASET)["query_id"],
        corpus_size=100_000,
        seed=0,                          # d38(c): fixed seed, recipe is a parameter
    )
    ms.materialize()
    ms.save("data")

ms_source = SnapshotDataset(MS_DATASET, path="data")
ms_corpus, ms_queries, ms_qrels = ms_source.corpus(), ms_source.queries(), ms_source.qrels()
print(f"corpus  {len(ms_corpus):,} passages "
      f"({ms_qrels.doc_id.nunique():,} judged-relevant, rest uniform distractors)")
print(f"queries {len(ms_queries):,} of {len(labels.rows_for(MS_DATASET)):,} composition rows "
      f"({len(ms_queries) / len(labels.rows_for(MS_DATASET)) * 100:.1f}% have dev qrels)")
print(f"qrels   {len(ms_qrels):,} judgments, grades {sorted(ms_qrels.relevance.unique())}")

corpus  100,000 passages (8,219 judged-relevant, rest uniform distractors)
queries 7,697 of 9,245 composition rows (83.3% have dev qrels)
qrels   8,228 judgments, grades [np.int64(1)]


## 12 — Index the 100K corpus

Same two named vector slots as every route collection. The one-time cost is
the dense pass over 100K passages (order of an hour on this machine); the
embedding cache makes re-runs cheap, and the upload skips when the collection
is already full.

In [22]:
ms_indexer = CorpusIndexer(
    client, MS_COLLECTION,
    embeddings=[dense_cfg, sparse_cfg],
    cache=EmbeddingCache("./.embedding_cache"),
)
ms_indexer.ensure_collection()

if client.count(MS_COLLECTION, exact=True).count >= len(ms_corpus):
    print(f"{MS_COLLECTION}: already indexed — skipping upload")
else:
    ms_indexer.upload(
        [CorpusDocument(**r) for r in ms_corpus.to_dict("records")], batch_size=64
    )
print(f"{MS_COLLECTION}: {client.count(MS_COLLECTION, exact=True).count:,} points")

msmarco_routes: already indexed — skipping upload
msmarco_routes: 100,000 points


## 13 — Label the lane

Same argmax rule as the nfcorpus anchor (d38e), so the two datasets differ by
exactly one variable — the index they are scored on. `RouteLabels.label`
narrows the 15,678 selection rows to the snapshot's queries via `QuerySubset`
and merges only this dataset's rows into `labels.parquet`. `min_relevance=1`
holds: the dev qrels are binary.

~7,700 queries × 3 routes; at nfcorpus throughput this is on the order of ten
minutes against local Qdrant.

In [23]:
ms_args = (client, MS_COLLECTION, dense_cfg, sparse_cfg)
ms_dense, ms_hybrid, ms_sparse = (
    DenseOnlyStrategy(*ms_args), PureRRFStrategy(*ms_args), SparseOnlyStrategy(*ms_args)
)

ms_labelled = labels.label(ms_source, ms_dense, ms_hybrid, ms_sparse, dataset=MS_DATASET)
print(f"labelled {len(ms_labelled):,} rows -> {labels.labels_path}")
show(ms_labelled.head(8).round(3))

goldenroutingbuilder:msmarco-passage-dev: 100%|██████████| 491/491 [00:27<00:00, 17.78it/s]

labelled 491 rows -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet


| dataset             |   query_id | query                                                | route       |   score |   score_dense_only |   score_pure_rrf |   score_sparse_only | shape         | metric_name               |   min_relevance | checkable   | cell                         | stage   | route_selected   |
|:--------------------|-----------:|:-----------------------------------------------------|:------------|--------:|-------------------:|-----------------:|--------------------:|:--------------|:--------------------------|----------------:|:------------|:-----------------------------|:--------|:-----------------|
| msmarco-passage-dev |    1048836 | who plays velma in scooby doo 2                      | dense_only  |   1     |              1     |            1     |               0.189 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |
| msmarco-passage-dev |    1048846 | what is option button style ?                        | dense_only  |   1     |              1     |            1     |               0.189 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |
| msmarco-passage-dev |        362 | . what are the president's main duties? explain each | dense_only  |   0.089 |              0.089 |            0     |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |
| msmarco-passage-dev |    1049663 | who sang 'convoy'                                    | dense_only  |   1     |              1     |            1     |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |
| msmarco-passage-dev |       1819 | A person with a pH below 7.35 is considered to be in | sparse_only |   1     |              0.1   |            0.189 |               1     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        | stopword_saturated_midlength | reused  | sparse_only      |
| msmarco-passage-dev |    1048922 | who presented casey affleck his oscar                | sparse_only |   1     |              1     |            1     |               1     | all_tied      | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |
| msmarco-passage-dev |     264284 | how long is peak time for morphine                   | dense_only  |   1     |              1     |            1     |               0     | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |
| msmarco-passage-dev |    1051095 | who sings hey good looking                           | dense_only  |   1     |              1     |            1     |               0.116 | routes_differ | 0.7*HitRate@1+0.3*NDCG@10 |               1 | True        |                              | control | nan              |

## 14 — Outcome shapes on a realistic index

Same three shapes as §7, now on a corpus that is 92% distractors. With a
median of **one** judged passage per query, `all_tied` requires all three
routes to place that same passage at the same effective rank, and
`routes_differ` means at least one route actually found it while another
did not — a much harder tie regime than nfcorpus's.

In [24]:
ms_shapes = (ms_labelled.groupby(["cell", "shape"]).size()
             .unstack(fill_value=0))
ms_shapes["total"] = ms_shapes.sum(axis=1)
show(ms_shapes.reset_index())

overall = ms_labelled["shape"].value_counts().rename_axis("shape").reset_index(name="rows")
overall["share"] = (overall.rows / len(ms_labelled) * 100).round(1).astype(str) + "%"
show(overall)

signal = ms_labelled[ms_labelled["shape"] == "routes_differ"]
dist = signal.route.value_counts().rename_axis("route").reset_index(name="rows")
dist["share"] = (dist.rows / len(signal) * 100).round(1).astype(str) + "%"
print(f"route distribution over the {len(signal):,} trainable rows:")
show(dist)
print(f"majority-class baseline: {dist.rows.iloc[0] / len(signal) * 100:.0f}% "
      f"(always predict {dist.route.iloc[0]})")

| cell                            |   all_tied |   all_zero |   routes_differ |   total |
|:--------------------------------|-----------:|-----------:|----------------:|--------:|
|                                 |         86 |          6 |              87 |     179 |
| acronym_inside_question         |          0 |          0 |               2 |       2 |
| bare_concept_token              |          0 |          0 |              40 |      40 |
| capsword_shape_ambiguity        |          0 |          0 |               3 |       3 |
| comparative_multi_entity        |          0 |          0 |               3 |       3 |
| conversational_courtesy_wrapper |          0 |          0 |               8 |       8 |
| deep_nesting_single_sentence    |          0 |          0 |               1 |       1 |
| high_morphological_variation    |          0 |          0 |              37 |      37 |
| keyword_telegram_short          |          0 |          0 |              43 |      43 |
| math_notation_present           |          0 |          0 |               2 |       2 |
| negation_bearing_question       |          0 |          0 |              21 |      21 |
| number_inside_natural_question  |          0 |          0 |              11 |      11 |
| relative_temporal_no_dates      |          0 |          0 |               8 |       8 |
| short_grammatical_question      |          0 |          0 |              60 |      60 |
| stopword_saturated_midlength    |          0 |          0 |              62 |      62 |
| verbose_grammatical_request     |          0 |          0 |               7 |       7 |
| wide_flat_enumeration           |          0 |          0 |               4 |       4 |

| shape         |   rows | share   |
|:--------------|-------:|:--------|
| routes_differ |    399 | 81.3%   |
| all_tied      |     86 | 17.5%   |
| all_zero      |      6 | 1.2%    |

route distribution over the 399 trainable rows:


| route       |   rows | share   |
|:------------|-------:|:--------|
| dense_only  |    330 | 82.7%   |
| sparse_only |     67 | 16.8%   |
| pure_rrf    |      2 | 0.5%    |

majority-class baseline: 83% (always predict dense_only)


## 15 — Where the golden set stands, two datasets in

The open label-form question (argmax one-hot vs the per-route score vector,
TODOS) turns on the **margin** — winner's score minus runner-up's. A fat
margin is a fact about retrieval; a thin one is a coin toss the objective
happened to break, and it flips when the encoder or corpus changes.
`decisive` counts rows with margin ≥ 0.06 *and* a rank-1 hit — roomier than
any NDCG-tail wiggle, and excluding "least bad" wins where every route missed.

nfcorpus margins were thin because 86% of its corpus is judged relevant to
*something* — two disjoint top-10s can both be right. msmarco's
median-1-relevant regime is the counter-test: either a route surfaced the one
judged passage or it scored zero, so ties require actually retrieving the
same passage at the same rank.

In [19]:
SCORES = ["score_dense_only", "score_pure_rrf", "score_sparse_only"]

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "median margin": round(differ.margin.median(), 3),
        "p90 margin": round(differ.margin.quantile(0.9), 3),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
    })
show(pd.DataFrame(rows))

coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} "
      f"({coverage.labelled.sum() / coverage.selected.sum() * 100:.1f}%)")

| dataset                              |   labelled |   routes_differ |   median margin |   p90 margin |   decisive | decisive share   |
|:-------------------------------------|-----------:|----------------:|----------------:|-------------:|-----------:|:-----------------|
| antique                              |        157 |             147 |           0.022 |        0.693 |         23 | 16%              |
| beir-nfcorpus                        |         12 |              12 |           0.725 |        0.779 |          9 | 75%              |
| bright-aops                          |          1 |               0 |         nan     |      nan     |          0 | nan%             |
| bright-biology                       |        102 |              75 |           0.029 |        0.764 |         17 | 23%              |
| bright-earth-science                 |        115 |              89 |           0.025 |        0.738 |         22 | 25%              |
| bright-economics                     |        103 |              56 |           0.029 |        0.394 |          8 | 14%              |
| bright-leetcode                      |          4 |               4 |           0.768 |        0.772 |          4 | 100%             |
| bright-pony                          |        101 |              79 |           0.022 |        0.724 |         16 | 20%              |
| bright-psychology                    |        100 |              59 |           0.029 |        0.754 |         10 | 17%              |
| bright-robotics                      |        101 |              58 |           0.026 |        0.304 |          6 | 10%              |
| bright-stackoverflow                 |        115 |              78 |           0.025 |        0.096 |         10 | 13%              |
| bright-sustainable-living            |        106 |              72 |           0.023 |        0.667 |         10 | 14%              |
| bright-theoremqa-questions           |          7 |               7 |           0.023 |        0.785 |          3 | 43%              |
| bright-theoremqa-theorems            |         76 |              22 |           0.038 |        0.768 |          6 | 27%              |
| clerc                                |       2556 |            1792 |           0.029 |        0.811 |        510 | 28%              |
| crumb-clinical-trial                 |         18 |              18 |           0.023 |        0.781 |          7 | 39%              |
| crumb-code-retrieval                 |        130 |              99 |           0.058 |        0.772 |         49 | 49%              |
| crumb-legal-qa                       |       2066 |            1369 |           0.041 |        0.811 |        373 | 27%              |
| crumb-paper-retrieval                |          4 |               4 |           0.356 |        0.715 |          2 | 50%              |
| crumb-set-operation-entity-retrieval |         35 |              30 |           0.056 |        0.737 |         14 | 47%              |
| crumb-stack-exchange                 |         12 |              12 |           0.78  |        0.846 |         10 | 83%              |
| crumb-theorem-retrieval              |          1 |               1 |           0     |        0     |          0 | 0%               |
| crumb-tip-of-the-tongue              |          2 |               2 |           0.02  |        0.025 |          0 | 0%               |
| dbpedia-entity                       |        256 |             237 |           0.013 |        0.055 |         20 | 8%               |
| freshstack-angular                   |        129 |             112 |           0.024 |        0.741 |         24 | 21%              |
| freshstack-godot                     |         98 |              61 |           0.024 |        0.685 |          8 | 13%              |
| freshstack-langchain                 |        201 |             167 |           0.023 |        0.734 |         32 | 19%              |
| freshstack-laravel                   |        184 |             147 |           0.028 |        0.729 |         29 | 20%              |
| freshstack-yolo                      |         57 |              34 |           0.03  |        0.743 |          8 | 24%              |
| gooaq                                |       7075 |            3184 |           0.029 |        0.811 |        798 | 25%              |
| limit                                |        277 |             257 |           0.789 |        0.805 |        133 | 52%              |
| lotte-technology-forum               |       1261 |            1109 |           0.02  |        0.709 |        170 | 15%              |
| lotte-technology-search              |        395 |             277 |           0.013 |        0.754 |         42 | 15%              |
| miracl-en-dev                        |        478 |             164 |           0     |        0.034 |         11 | 7%               |
| msmarco-passage-dev                  |       3816 |            1843 |           0     |        0.811 |        538 | 29%              |
| orcas                                |       7547 |            4613 |           0.018 |        0.811 |        953 | 21%              |
| quest                                |       1118 |             808 |           0.022 |        0.72  |        117 | 14%              |
| rarb-code                            |        193 |             169 |           0.087 |        0.811 |         81 | 48%              |
| rarb-math                            |        440 |             357 |           0.811 |        0.811 |        206 | 58%              |
| scirgen-geo-en                       |       8030 |            2608 |           0.029 |        0.811 |        384 | 15%              |
| trec-dl-2022                         |         51 |              47 |           0.037 |        0.767 |         16 | 34%              |
| webfaq-eng                           |       8612 |            2664 |           0     |        0.811 |        614 | 23%              |

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| webfaq-eng                           |       9345 |       8612 |             0 |          733 |            2664 |       5173 |        775 |
| scirgen-geo-en                       |       8288 |       8030 |             0 |          258 |            2608 |       1228 |       4194 |
| orcas                                |       7782 |       7547 |             0 |          235 |            4613 |       2154 |        780 |
| gooaq                                |       7296 |       7075 |             0 |          221 |            3184 |       3206 |        685 |
| msmarco-passage-dev                  |       9245 |       3816 |             0 |         5429 |            1843 |       1874 |         99 |
| clerc                                |       2612 |       2556 |             0 |           56 |            1792 |        715 |         49 |
| crumb-legal-qa                       |       2209 |       2066 |             0 |          143 |            1369 |         71 |        626 |
| lotte-technology-forum               |       1584 |       1261 |             0 |          323 |            1109 |         42 |        110 |
| quest                                |       1432 |       1118 |             0 |          314 |             808 |          7 |        303 |
| miracl-en-dev                        |        576 |        478 |             0 |           98 |             164 |        314 |          0 |
| rarb-math                            |        638 |        440 |             0 |          198 |             357 |         64 |         19 |
| lotte-technology-search              |        472 |        395 |             0 |           77 |             277 |        108 |         10 |
| limit                                |        277 |        277 |             0 |            0 |             257 |          0 |         20 |
| dbpedia-entity                       |        280 |        256 |             0 |           24 |             237 |         17 |          2 |
| freshstack-langchain                 |        546 |        201 |             0 |          345 |             167 |          1 |         33 |
| rarb-code                            |        265 |        193 |             0 |           72 |             169 |          4 |         20 |
| freshstack-laravel                   |        482 |        184 |             0 |          298 |             147 |          3 |         34 |
| antique                              |        190 |        157 |             0 |           33 |             147 |          0 |         10 |
| crumb-code-retrieval                 |        273 |        130 |             0 |          143 |              99 |          1 |         30 |
| freshstack-angular                   |        347 |        129 |             0 |          218 |             112 |          0 |         17 |
| bright-earth-science                 |        310 |        115 |             0 |          195 |              89 |          9 |         17 |
| bright-stackoverflow                 |        302 |        115 |             0 |          187 |              78 |          6 |         31 |
| bright-sustainable-living            |        334 |        106 |             0 |          228 |              72 |          2 |         32 |
| bright-economics                     |        321 |        103 |             0 |          218 |              56 |          6 |         41 |
| bright-biology                       |        264 |        102 |             0 |          162 |              75 |          4 |         23 |
| bright-robotics                      |        284 |        101 |             0 |          183 |              58 |          2 |         41 |
| bright-pony                          |        190 |        101 |             0 |           89 |              79 |          0 |         22 |
| bright-psychology                    |        314 |        100 |             0 |          214 |              59 |          4 |         37 |
| freshstack-godot                     |        271 |         98 |             0 |          173 |              61 |          1 |         36 |
| bright-theoremqa-theorems            |        185 |         76 |             0 |          109 |              22 |          0 |         54 |
| freshstack-yolo                      |        163 |         57 |             0 |          106 |              34 |          2 |         21 |
| trec-dl-2022                         |        381 |         51 |             0 |          330 |              47 |          0 |          4 |
| crumb-set-operation-entity-retrieval |         39 |         35 |             0 |            4 |              30 |          0 |          5 |
| crumb-clinical-trial                 |         56 |         18 |             0 |           38 |              18 |          0 |          0 |
| crumb-stack-exchange                 |         37 |         12 |             0 |           25 |              12 |          0 |          0 |
| beir-nfcorpus                        |         12 |         12 |             0 |            0 |              12 |          0 |          0 |
| bright-theoremqa-questions           |         17 |          7 |             0 |           10 |               7 |          0 |          0 |
| crumb-paper-retrieval                |         14 |          4 |             0 |           10 |               4 |          0 |          0 |
| bright-leetcode                      |         12 |          4 |             0 |            8 |               4 |          0 |          0 |
| crumb-tip-of-the-tongue              |          5 |          2 |             0 |            3 |               2 |          0 |          0 |
| crumb-theorem-retrieval              |          2 |          1 |             0 |            1 |               1 |          0 |          0 |
| bright-aops                          |          1 |          1 |             0 |            0 |               0 |          0 |          1 |

labelled 46,142 of 57,653 (80.0%)


## 16 — Readiness: every lane materialized, only indexing remains

The two-pass acquisition (qrels, then corpus) is complete — all 42 lanes were
built by `materialize_corpora.py`, so `cell_selection` needs no fetch. What's
left is indexing each corpus into Qdrant, then labelling.

This cell is the progress bar and defines the helpers the sweep reuses:
`collection_of` maps a composition key to its route collection (with the one
pre-convention name, `msmarco_routes`, overridden so the 100K index is reused
not rebuilt), and `datasets` is every selected dataset ordered **cheapest
corpus first** — the sweep's order.

In [12]:
from pyarrow.parquet import ParquetFile

from hybrid_search_rrf_dataset.lanes import LANES

# the two anchor collections predate the {source.name}_routes convention
# (§4, §12); override so the sweep reuses their indexes, not re-embeds under
# new names.
COLLECTION_OVERRIDE = {
    "beir-nfcorpus": "nfcorpus_routes",
    "msmarco-passage-dev": "msmarco_routes",
}


def source_name(key: str) -> str:
    return LANES[key].source.name if key in LANES else key


def collection_of(key: str) -> str:
    return COLLECTION_OVERRIDE.get(key, f"{source_name(key)}_routes")


def corpus_rows(key: str) -> int | None:
    path = Path("data") / source_name(key) / "corpus.parquet"
    return ParquetFile(path).metadata.num_rows if path.exists() else None


# every selected dataset, cheapest corpus first — the sweep's order
datasets = sorted(
    selection["dataset"].astype(str).unique(),
    key=lambda k: corpus_rows(k) if corpus_rows(k) is not None else float("inf"),
)
live = {c.name for c in client.get_collections().collections}
done = set(labels.load()["dataset"].unique()) if labels.labels_path.exists() else set()

ready = pd.DataFrame(
    [
        {
            "dataset": k,
            "corpus_rows": corpus_rows(k),
            "indexed": collection_of(k) in live,
            "labelled": k in done,
        }
        for k in datasets
    ]
)
show(ready)
print(
    f"datasets {len(ready)} | corpus on disk {ready.corpus_rows.notna().sum()} "
    f"| indexed {int(ready.indexed.sum())} | labelled {int(ready.labelled.sum())}"
)

| dataset                              |   corpus_rows | indexed   | labelled   |
|:-------------------------------------|--------------:|:----------|:-----------|
| scirgen-geo-en                       |          3354 | True      | True       |
| beir-nfcorpus                        |          3633 | True      | True       |
| bright-pony                          |          7894 | True      | True       |
| lotte-technology-search              |         10000 | True      | True       |
| crumb-legal-qa                       |         10000 | True      | True       |
| bright-biology                       |         10000 | True      | True       |
| bright-earth-science                 |         10000 | True      | True       |
| bright-theoremqa-questions           |         10000 | True      | True       |
| crumb-stack-exchange                 |         10000 | True      | True       |
| bright-economics                     |         10000 | True      | True       |
| bright-psychology                    |         10000 | True      | True       |
| bright-robotics                      |         10000 | True      | True       |
| bright-stackoverflow                 |         10000 | True      | True       |
| bright-sustainable-living            |         10000 | True      | True       |
| bright-theoremqa-theorems            |         10000 | True      | True       |
| freshstack-angular                   |         10000 | True      | True       |
| freshstack-godot                     |         10000 | True      | True       |
| freshstack-langchain                 |         10000 | True      | True       |
| freshstack-laravel                   |         10000 | True      | True       |
| bright-leetcode                      |         10000 | True      | True       |
| crumb-tip-of-the-tongue              |         10000 | True      | True       |
| crumb-theorem-retrieval              |         10000 | True      | True       |
| freshstack-yolo                      |         10000 | True      | True       |
| bright-aops                          |         10000 | True      | True       |
| miracl-en-dev                        |         11480 | True      | True       |
| webfaq-eng                           |         26140 | True      | True       |
| crumb-paper-retrieval                |         28495 | True      | True       |
| trec-dl-2022                         |         30000 | True      | True       |
| orcas                                |         30530 | True      | True       |
| rarb-math                            |         31595 | True      | True       |
| antique                              |         32430 | True      | True       |
| limit                                |         50000 | True      | True       |
| crumb-set-operation-entity-retrieval |         57730 | True      | True       |
| quest                                |         72080 | True      | True       |
| dbpedia-entity                       |         74385 | True      | True       |
| lotte-technology-forum               |         79450 | True      | True       |
| msmarco-passage-dev                  |        100000 | True      | True       |
| gooaq                                |        100000 | True      | True       |
| rarb-code                            |        100000 | True      | True       |
| clerc                                |        100000 | True      | True       |
| crumb-clinical-trial                 |        100000 | True      | True       |
| crumb-code-retrieval                 |        119976 | True      | True       |

datasets 42 | corpus on disk 42 | indexed 42 | labelled 42


In [13]:
coverage = labels.coverage()
show(coverage)
print(f"selected {coverage.selected.sum():,} | "
      f"labelled {coverage.labelled.sum():,} | "
      f"qrels_ready {coverage.qrels_ready.sum():,} | "
      f"unlabelled {coverage.unlabelled.sum():,}")

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| scirgen-geo-en                       |       8288 |       8030 |             0 |          258 |            2608 |       1228 |       4194 |
| crumb-legal-qa                       |       2209 |       2066 |             0 |          143 |            1369 |         71 |        626 |
| webfaq-eng                           |       9345 |       2065 |             0 |         7280 |             613 |       1391 |         61 |
| lotte-technology-forum               |       1584 |       1261 |             0 |          323 |            1109 |         42 |        110 |
| orcas                                |       7782 |       1208 |             0 |         6574 |             706 |        371 |        131 |
| quest                                |       1432 |       1118 |             0 |          314 |             808 |          7 |        303 |
| msmarco-passage-dev                  |       9245 |        491 |             0 |         8754 |             358 |        127 |          6 |
| miracl-en-dev                        |        576 |        478 |             0 |           98 |             164 |        314 |          0 |
| rarb-math                            |        638 |        440 |             0 |          198 |             357 |         64 |         19 |
| clerc                                |       2612 |        396 |             0 |         2216 |             275 |         46 |         75 |
| lotte-technology-search              |        472 |        395 |             0 |           77 |             277 |        108 |         10 |
| limit                                |        277 |        277 |             0 |            0 |             257 |          0 |         20 |
| dbpedia-entity                       |        280 |        256 |             0 |           24 |             237 |         17 |          2 |
| freshstack-langchain                 |        546 |        201 |             0 |          345 |             167 |          1 |         33 |
| rarb-code                            |        265 |        193 |             0 |           72 |             169 |          4 |         20 |
| freshstack-laravel                   |        482 |        184 |             0 |          298 |             147 |          3 |         34 |
| antique                              |        190 |        157 |             0 |           33 |             147 |          0 |         10 |
| crumb-code-retrieval                 |        273 |        130 |             0 |          143 |              99 |          1 |         30 |
| freshstack-angular                   |        347 |        129 |             0 |          218 |             112 |          0 |         17 |
| gooaq                                |       7296 |        119 |             0 |         7177 |              62 |         42 |         15 |
| bright-earth-science                 |        310 |        115 |             0 |          195 |              89 |          9 |         17 |
| bright-stackoverflow                 |        302 |        115 |             0 |          187 |              78 |          6 |         31 |
| bright-sustainable-living            |        334 |        106 |             0 |          228 |              72 |          2 |         32 |
| bright-economics                     |        321 |        103 |             0 |          218 |              56 |          6 |         41 |
| bright-biology                       |        264 |        102 |             0 |          162 |              75 |          4 |         23 |
| bright-robotics                      |        284 |        101 |             0 |          183 |              58 |          2 |         41 |
| bright-pony                          |        190 |        101 |             0 |           89 |              79 |          0 |         22 |
| bright-psychology                    |        314 |        100 |             0 |          214 |              59 |          4 |         37 |
| freshstack-godot                     |        271 |         98 |             0 |          173 |              61 |          1 |         36 |
| bright-theoremqa-theorems            |        185 |         76 |             0 |          109 |              22 |          0 |         54 |
| freshstack-yolo                      |        163 |         57 |             0 |          106 |              34 |          2 |         21 |
| trec-dl-2022                         |        381 |         51 |             0 |          330 |              47 |          0 |          4 |
| crumb-set-operation-entity-retrieval |         39 |         35 |             0 |            4 |              30 |          0 |          5 |
| crumb-clinical-trial                 |         56 |         18 |             0 |           38 |              18 |          0 |          0 |
| crumb-stack-exchange                 |         37 |         12 |             0 |           25 |              12 |          0 |          0 |
| beir-nfcorpus                        |         12 |         12 |             0 |            0 |              12 |          0 |          0 |
| bright-theoremqa-questions           |         17 |          7 |             0 |           10 |               7 |          0 |          0 |
| crumb-paper-retrieval                |         14 |          4 |             0 |           10 |               4 |          0 |          0 |
| bright-leetcode                      |         12 |          4 |             0 |            8 |               4 |          0 |          0 |
| crumb-tip-of-the-tongue              |          5 |          2 |             0 |            3 |               2 |          0 |          0 |
| crumb-theorem-retrieval              |          2 |          1 |             0 |            1 |               1 |          0 |          0 |
| bright-aops                          |          1 |          1 |             0 |            0 |               0 |          0 |          1 |

selected 57,653 | labelled 20,815 | qrels_ready 0 | unlabelled 36,838


## 17 — Sweep: index and label every remaining lane, cheapest first

One loop over all 42 datasets in `datasets` order (§16, size-ascending), so the
cheap lanes label first and a long dense pass never blocks quick wins. Per lane:
read the on-disk corpus → index into its route collection if not already full →
label with the argmax objective. Every step idempotent: a full collection skips
the upload, and a dataset already in `labels.parquet` is skipped whole.

- **`min_relevance` is per-lane** from `LANES` — antique binarizes at grade 3
  (its level 2 is "does not answer"), trec-dl at 2, the rest at 1. A fresh
  `RouteLabels` per lane carries the right threshold into the label and its
  stored `min_relevance`.
- **Corpus sizing already happened** in `materialize_corpora.py` (the d39 20/80
  `CorpusRecipe`, per-lane exceptions pinned in `LANES`), so the sweep only reads
  snapshots — no `materialize()` here.
- **BRIGHT `excluded_ids`** ride on each snapshot's `excluded.parquet` and are
  applied per query at scoring time inside `RouteLabels.label`.
- **One bad lane never aborts the sweep** — a lane whose selected queries have
  no judgments is caught and reported, and the loop continues.

> Starting a clean golden set? `labels.parquet` may still hold rows from the old
> `selection.parquet` query set. Delete it first
> (`rm data/route_labels/labels.parquet`) so labels aren't a mix of two
> compositions; the sweep then rebuilds every lane.

In [42]:
already = set(labels.load()["dataset"].unique()) if labels.labels_path.exists() else set()

for key in datasets:                         # size-ascending, from §16
    if key in already:
        print(f"{key:38s} labelled — skip")
        continue

    name = source_name(key)
    src = SnapshotDataset(name, path="data")
    corpus = src.corpus()
    collection = collection_of(key)

    lane_indexer = CorpusIndexer(
        client, collection,
        embeddings=[dense_cfg, sparse_cfg],
        cache=EmbeddingCache("./.embedding_cache"),
    )
    lane_indexer.ensure_collection()
    if client.count(collection, exact=True).count < len(corpus):
        lane_indexer.upload(
            [CorpusDocument(**r) for r in corpus.to_dict("records")], batch_size=64
        )

    # per-lane threshold; a fresh RouteLabels shares the same labels.parquet
    min_rel = LANES[key].min_relevance if key in LANES else 1
    lane_labels = RouteLabels(selection, objective=RouterObjective(min_relevance=min_rel))

    strat_args = (client, collection, dense_cfg, sparse_cfg)
    try:
        out = lane_labels.label(
            src,
            DenseOnlyStrategy(*strat_args),
            PureRRFStrategy(*strat_args),
            SparseOnlyStrategy(*strat_args),
            dataset=key,
        )
    except ValueError as error:              # no judged queries in this lane's selection
        print(f"{key:38s} SKIPPED: {error}")
        continue

    shp = out["shape"].value_counts()
    print(f"{key:38s} corpus {len(corpus):>7,}  min_rel {min_rel}  "
          f"labelled {len(out):>6,}  differ {shp.get('routes_differ', 0):,} "
          f"| tied {shp.get('all_tied', 0):,} | zero {shp.get('all_zero', 0):,}")

scirgen-geo-en                         labelled — skip
beir-nfcorpus                          labelled — skip
bright-pony                            labelled — skip
lotte-technology-search                labelled — skip
crumb-legal-qa                         labelled — skip
bright-biology                         labelled — skip
bright-earth-science                   labelled — skip
bright-theoremqa-questions             labelled — skip
crumb-stack-exchange                   labelled — skip
bright-economics                       labelled — skip
bright-psychology                      labelled — skip
bright-robotics                        labelled — skip
bright-stackoverflow                   labelled — skip
bright-sustainable-living              labelled — skip
bright-theoremqa-theorems              labelled — skip
freshstack-angular                     labelled — skip
freshstack-godot                       labelled — skip
freshstack-langchain                   labelled — skip
freshstack

goldenroutingbuilder:quest: 100%|██████████| 1118/1118 [01:17<00:00, 14.41it/s]


quest                                  corpus  72,080  min_rel 1  labelled  1,118  differ 789 | tied 5 | zero 324


goldenroutingbuilder:dbpedia-entity: 100%|██████████| 256/256 [00:05<00:00, 43.67it/s]


dbpedia-entity                         corpus  74,385  min_rel 1  labelled    256  differ 241 | tied 13 | zero 2


...

clerc                                  corpus 100,000  min_rel 1  labelled    396  differ 230 | tied 43 | zero 123


goldenroutingbuilder:crumb-clinical-trial: 100%|██████████| 18/18 [00:03<00:00,  5.01it/s]


crumb-clinical-trial                   corpus 100,000  min_rel 1  labelled     18  differ 18 | tied 0 | zero 0


goldenroutingbuilder:crumb-code-retrieval: 100%|██████████| 130/130 [00:10<00:00, 12.35it/s]

crumb-code-retrieval                   corpus 119,976  min_rel 1  labelled    130  differ 109 | tied 0 | zero 21


In [43]:
coverage = labels.coverage()
show(coverage[coverage.labelled > 0])
print(f"labelled {coverage.labelled.sum():,} of {coverage.selected.sum():,} | "
      f"qrels_ready {coverage.qrels_ready.sum():,}")

done = labels.load()
done["margin"] = done[SCORES].max(axis=1) - done[SCORES].apply(
    lambda r: sorted(r)[-2], axis=1
)
done["hit"] = done[SCORES].max(axis=1) >= 0.7

rows = []
for ds, group in done.groupby("dataset"):
    differ = group[group["shape"] == "routes_differ"]
    decisive = (differ.margin >= 0.06) & differ.hit
    dist = differ.route.value_counts()
    rows.append({
        "dataset": ds,
        "labelled": len(group),
        "routes_differ": len(differ),
        "decisive": int(decisive.sum()),
        "decisive share": f"{decisive.mean() * 100:.0f}%",
        "median margin": round(differ.margin.median(), 3) if len(differ) else None,
        "top route": dist.index[0] if len(dist) else None,
        "top share": f"{dist.iloc[0] / len(differ) * 100:.0f}%" if len(differ) else None,
    })
show(pd.DataFrame(rows).sort_values("labelled", ascending=False))

Failed to reload module 'augmentation.loop' from file '/Users/andrei/projects/hybrid-search-rrf-dataset/src/augmentation/loop.py'
Traceback (most recent call last):
  File "/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/Users/andrei/.pyenv/versions/3.12.13/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/Users/andrei/projects/hybrid-search-rrf-dataset/src/augmentation/loop.py", 

| dataset                              |   selected |   labelled |   qrels_ready |   unlabelled |   routes_differ |   all_tied |   all_zero |
|:-------------------------------------|-----------:|-----------:|--------------:|-------------:|----------------:|-----------:|-----------:|
| scirgen-geo-en                       |       8288 |       8030 |             0 |          258 |            2617 |       1073 |       4340 |
| crumb-legal-qa                       |       2209 |       2066 |             0 |          143 |            1356 |         38 |        672 |
| webfaq-eng                           |       9345 |       2065 |             0 |         7280 |             724 |       1236 |        105 |
| lotte-technology-forum               |       1584 |       1261 |             0 |          323 |            1094 |         33 |        134 |
| orcas                                |       7782 |       1208 |             0 |         6574 |             711 |        342 |        155 |
| quest                                |       1432 |       1118 |             0 |          314 |             789 |          5 |        324 |
| msmarco-passage-dev                  |       9245 |        491 |             0 |         8754 |             399 |         86 |          6 |
| miracl-en-dev                        |        576 |        478 |             0 |           98 |             203 |        275 |          0 |
| rarb-math                            |        638 |        440 |             0 |          198 |             394 |         25 |         21 |
| clerc                                |       2612 |        396 |             0 |         2216 |             230 |         43 |        123 |
| lotte-technology-search              |        472 |        395 |             0 |           77 |             285 |         86 |         24 |
| limit                                |        277 |        277 |             0 |            0 |             259 |          0 |         18 |
| dbpedia-entity                       |        280 |        256 |             0 |           24 |             241 |         13 |          2 |
| freshstack-langchain                 |        546 |        201 |             0 |          345 |             167 |          2 |         32 |
| rarb-code                            |        265 |        193 |             0 |           72 |             172 |          0 |         21 |
| freshstack-laravel                   |        482 |        184 |             0 |          298 |             150 |          1 |         33 |
| antique                              |        190 |        157 |             0 |           33 |             149 |          0 |          8 |
| crumb-code-retrieval                 |        273 |        130 |             0 |          143 |             109 |          0 |         21 |
| freshstack-angular                   |        347 |        129 |             0 |          218 |             113 |          0 |         16 |
| gooaq                                |       7296 |        119 |             0 |         7177 |              62 |         38 |         19 |
| bright-earth-science                 |        310 |        115 |             0 |          195 |              87 |          9 |         19 |
| bright-stackoverflow                 |        302 |        115 |             0 |          187 |              78 |          5 |         32 |
| bright-sustainable-living            |        334 |        106 |             0 |          228 |              67 |          2 |         37 |
| bright-economics                     |        321 |        103 |             0 |          218 |              58 |          6 |         39 |
| bright-biology                       |        264 |        102 |             0 |          162 |              74 |          2 |         26 |
| bright-robotics                      |        284 |        101 |             0 |          183 |              53 |          3 |         45 |
| bright-pony                          |        190 |        101 |             0 |           89 |              72 |          0 |         29 |
| bright-psychology                    |        314 |        100 |             0 |          214 |              59 |          4 |         37 |
| freshstack-godot                     |        271 |         98 |             0 |          173 |              59 |          1 |         38 |
| bright-theoremqa-theorems            |        185 |         76 |             0 |          109 |              19 |          0 |         57 |
| freshstack-yolo                      |        163 |         57 |             0 |          106 |              34 |          0 |         23 |
| trec-dl-2022                         |        381 |         51 |             0 |          330 |              48 |          0 |          3 |
| crumb-set-operation-entity-retrieval |         39 |         35 |             0 |            4 |              29 |          0 |          6 |
| crumb-clinical-trial                 |         56 |         18 |             0 |           38 |              18 |          0 |          0 |
| crumb-stack-exchange                 |         37 |         12 |             0 |           25 |              12 |          0 |          0 |
| beir-nfcorpus                        |         12 |         12 |             0 |            0 |              12 |          0 |          0 |
| bright-theoremqa-questions           |         17 |          7 |             0 |           10 |               7 |          0 |          0 |
| crumb-paper-retrieval                |         14 |          4 |             0 |           10 |               4 |          0 |          0 |
| bright-leetcode                      |         12 |          4 |             0 |            8 |               4 |          0 |          0 |
| crumb-tip-of-the-tongue              |          5 |          2 |             0 |            3 |               2 |          0 |          0 |
| crumb-theorem-retrieval              |          2 |          1 |             0 |            1 |               1 |          0 |          0 |
| bright-aops                          |          1 |          1 |             0 |            0 |               0 |          0 |          1 |

labelled 20,815 of 57,653 | qrels_ready 0


| dataset                              |   labelled |   routes_differ |   decisive | decisive share   |   median margin | top route   | top share   |
|:-------------------------------------|-----------:|----------------:|-----------:|:-----------------|----------------:|:------------|:------------|
| scirgen-geo-en                       |       8030 |            2617 |        382 | 15%              |           0.029 | dense_only  | 50%         |
| crumb-legal-qa                       |       2066 |            1356 |        456 | 34%              |           0.058 | dense_only  | 88%         |
| webfaq-eng                           |       2065 |             724 |        166 | 23%              |           0     | dense_only  | 84%         |
| lotte-technology-forum               |       1261 |            1094 |        187 | 17%              |           0.023 | dense_only  | 63%         |
| orcas                                |       1208 |             711 |        138 | 19%              |           0.018 | dense_only  | 71%         |
| quest                                |       1118 |             789 |        107 | 14%              |           0.021 | sparse_only | 42%         |
| msmarco-passage-dev                  |        491 |             399 |        293 | 73%              |           0.811 | dense_only  | 83%         |
| miracl-en-dev                        |        478 |             203 |         14 | 7%               |           0     | dense_only  | 54%         |
| rarb-math                            |        440 |             394 |        307 | 78%              |           0.811 | dense_only  | 63%         |
| clerc                                |        396 |             230 |         36 | 16%              |           0.034 | sparse_only | 73%         |
| lotte-technology-search              |        395 |             285 |         42 | 15%              |           0.018 | dense_only  | 71%         |
| limit                                |        277 |             259 |        135 | 52%              |           0.781 | sparse_only | 100%        |
| dbpedia-entity                       |        256 |             241 |         22 | 9%               |           0.014 | dense_only  | 45%         |
| freshstack-langchain                 |        201 |             167 |         37 | 22%              |           0.026 | dense_only  | 41%         |
| rarb-code                            |        193 |             172 |         80 | 47%              |           0.06  | dense_only  | 99%         |
| freshstack-laravel                   |        184 |             150 |         27 | 18%              |           0.024 | sparse_only | 45%         |
| antique                              |        157 |             149 |         20 | 13%              |           0.022 | dense_only  | 64%         |
| crumb-code-retrieval                 |        130 |             109 |         70 | 64%              |           0.724 | dense_only  | 79%         |
| freshstack-angular                   |        129 |             113 |         23 | 20%              |           0.026 | dense_only  | 42%         |
| gooaq                                |        119 |              62 |         12 | 19%              |           0.037 | dense_only  | 90%         |
| bright-stackoverflow                 |        115 |              78 |         12 | 15%              |           0.029 | sparse_only | 46%         |
| bright-earth-science                 |        115 |              87 |         19 | 22%              |           0.02  | dense_only  | 46%         |
| bright-sustainable-living            |        106 |              67 |          7 | 10%              |           0.016 | dense_only  | 55%         |
| bright-economics                     |        103 |              58 |          5 | 9%               |           0.029 | dense_only  | 64%         |
| bright-biology                       |        102 |              74 |         14 | 19%              |           0.023 | sparse_only | 54%         |
| bright-pony                          |        101 |              72 |         17 | 24%              |           0.021 | sparse_only | 76%         |
| bright-robotics                      |        101 |              53 |          9 | 17%              |           0.029 | sparse_only | 45%         |
| bright-psychology                    |        100 |              59 |          5 | 8%               |           0.025 | dense_only  | 44%         |
| freshstack-godot                     |         98 |              59 |         11 | 19%              |           0.027 | dense_only  | 46%         |
| bright-theoremqa-theorems            |         76 |              19 |          2 | 11%              |           0.018 | dense_only  | 58%         |
| freshstack-yolo                      |         57 |              34 |         10 | 29%              |           0.038 | sparse_only | 56%         |
| trec-dl-2022                         |         51 |              48 |         14 | 29%              |           0.028 | dense_only  | 75%         |
| crumb-set-operation-entity-retrieval |         35 |              29 |         16 | 55%              |           0.702 | sparse_only | 52%         |
| crumb-clinical-trial                 |         18 |              18 |         14 | 78%              |           0.71  | dense_only  | 83%         |
| crumb-stack-exchange                 |         12 |              12 |         12 | 100%             |           0.788 | dense_only  | 75%         |
| beir-nfcorpus                        |         12 |              12 |         11 | 92%              |           0.739 | dense_only  | 50%         |
| bright-theoremqa-questions           |          7 |               7 |          5 | 71%              |           0.759 | dense_only  | 71%         |
| bright-leetcode                      |          4 |               4 |          3 | 75%              |           0.756 | dense_only  | 75%         |
| crumb-paper-retrieval                |          4 |               4 |          2 | 50%              |           0.351 | dense_only  | 75%         |
| crumb-tip-of-the-tongue              |          2 |               2 |          1 | 50%              |           0.394 | dense_only  | 100%        |
| bright-aops                          |          1 |               0 |          0 | nan%             |         nan     | nan         | nan         |
| crumb-theorem-retrieval              |          1 |               1 |          0 | 0%               |           0     | sparse_only | 100%        |

## 18 — How much is routing worth? The headroom readout

Three levels, same labelled rows:

1. **One global constant** — always answer with the single best method, everywhere.
2. **Best constant per collection** — someone tells you, per dataset, which one method
   works best there, and you follow it blindly.
3. **Per-query oracle** — a fortune-teller picks the best method for every individual
   query. This is the ceiling.

The 1→2 gap is what *knowing your collection* is worth; 2→3 is what *judging each query*
adds. Together they are the business case for a router.

**Read it with the fine print, always:**

- The oracle is a **ceiling, not an achievement**. Published attempts at this kind of
  per-query selection achieved ~4% at best — a router capturing even half of our ceiling
  would be exceptional.
- The pooled number depends on the **dataset mix**, which was never designed for routing:
  these queries were composed for feature diversity. **This is not an optimal routing
  dataset**, and its pooled headroom must not be quoted without this caveat.
- Judgment holes are unmeasured and bias scores against dense; every number is pinned to
  the current embedding stack (bge-small).
- **One-sided decisiveness is not headroom.** `limit` is 43% decisive with ~0 headroom:
  one method wins every decisive row there, so the constant already captures it — visible
  in the winner table below.

Decisive = the winner put a relevant document at rank 1 and the runner-up did not; the
margin threshold is derived from the objective's weights, never hand-typed.

Ratified decision and full caveat list: SPEC.md decision 44.

In [44]:
show(labels.headroom_decomposition().round(3))

headroom = labels.headroom()
show(headroom)

winners = labels.decisive_winners()
show(winners)

pooled = headroom[headroom.dataset == "POOLED"].iloc[0]
mix = headroom[headroom.dataset != "POOLED"].nlargest(1, "labelled").iloc[0]
print(f"ceiling over one global constant: +{pooled.headroom_pct}% "
      f"({pooled.constant:.3f} -> {pooled.oracle:.3f})")
print(f"lane-mix warning: {mix.dataset} alone is "
      f"{mix.labelled / pooled.labelled * 100:.0f}% of labelled rows "
      f"and its own ceiling is only +{mix.headroom_pct}%")

| level                            |   score |   gain_vs_previous_pct |
|:---------------------------------|--------:|-----------------------:|
| one global constant (dense_only) |   0.404 |                    nan |
| best constant per collection     |   0.437 |                      8 |
| per-query oracle (ceiling)       |   0.494 |                     13 |

| dataset                              |   labelled |   oracle | best_constant   |   constant |   headroom |   headroom_pct |   decisive_share |   all_zero_share |
|:-------------------------------------|-----------:|---------:|:----------------|-----------:|-----------:|---------------:|-----------------:|-----------------:|
| antique                              |        157 |    0.722 | dense_only      |      0.646 |      0.076 |           11.7 |            0.102 |            0.051 |
| beir-nfcorpus                        |         12 |    0.836 | dense_only      |      0.449 |      0.387 |           86.3 |            0.917 |            0     |
| bright-aops                          |          1 |    0     | dense_only      |      0     |      0     |            0   |            0     |            1     |
| bright-biology                       |        102 |    0.416 | sparse_only     |      0.335 |      0.082 |           24.5 |            0.127 |            0.255 |
| bright-earth-science                 |        115 |    0.596 | pure_rrf        |      0.467 |      0.129 |           27.5 |            0.157 |            0.165 |
| bright-economics                     |        103 |    0.301 | pure_rrf        |      0.256 |      0.045 |           17.7 |            0.039 |            0.379 |
| bright-leetcode                      |          4 |    0.917 | dense_only      |      0.696 |      0.221 |           31.7 |            0.75  |            0     |
| bright-pony                          |        101 |    0.225 | sparse_only     |      0.191 |      0.033 |           17.5 |            0.168 |            0.287 |
| bright-psychology                    |        100 |    0.32  | pure_rrf        |      0.281 |      0.039 |           14   |            0.04  |            0.37  |
| bright-robotics                      |        101 |    0.281 | pure_rrf        |      0.217 |      0.063 |           29.2 |            0.089 |            0.446 |
| bright-stackoverflow                 |        115 |    0.425 | pure_rrf        |      0.35  |      0.075 |           21.3 |            0.096 |            0.278 |
| bright-sustainable-living            |        106 |    0.362 | pure_rrf        |      0.316 |      0.046 |           14.6 |            0.047 |            0.349 |
| bright-theoremqa-questions           |          7 |    0.935 | dense_only      |      0.713 |      0.222 |           31.2 |            0.714 |            0     |
| bright-theoremqa-theorems            |         76 |    0.109 | pure_rrf        |      0.084 |      0.025 |           30.3 |            0.026 |            0.75  |
| clerc                                |        396 |    0.39  | sparse_only     |      0.337 |      0.053 |           15.7 |            0.091 |            0.311 |
| crumb-clinical-trial                 |         18 |    0.884 | dense_only      |      0.753 |      0.131 |           17.4 |            0.556 |            0     |
| crumb-code-retrieval                 |        130 |    0.671 | dense_only      |      0.537 |      0.134 |           25   |            0.446 |            0.162 |
| crumb-legal-qa                       |       2066 |    0.366 | dense_only      |      0.34  |      0.026 |            7.6 |            0.213 |            0.325 |
| crumb-paper-retrieval                |          4 |    0.867 | dense_only      |      0.694 |      0.173 |           24.9 |            0.5   |            0     |
| crumb-set-operation-entity-retrieval |         35 |    0.631 | sparse_only     |      0.378 |      0.253 |           67.1 |            0.457 |            0.171 |
| crumb-stack-exchange                 |         12 |    0.929 | dense_only      |      0.695 |      0.234 |           33.7 |            1     |            0     |
| crumb-theorem-retrieval              |          1 |    0.884 | pure_rrf        |      0.884 |      0     |            0   |            0     |            0     |
| crumb-tip-of-the-tongue              |          2 |    0.844 | dense_only      |      0.844 |      0     |            0   |            0.5   |            0     |
| dbpedia-entity                       |        256 |    0.889 | pure_rrf        |      0.84  |      0.048 |            5.7 |            0.047 |            0.008 |
| freshstack-angular                   |        129 |    0.45  | pure_rrf        |      0.333 |      0.117 |           35.1 |            0.155 |            0.124 |
| freshstack-godot                     |         98 |    0.341 | pure_rrf        |      0.257 |      0.084 |           32.5 |            0.112 |            0.388 |
| freshstack-langchain                 |        201 |    0.428 | pure_rrf        |      0.308 |      0.12  |           38.9 |            0.154 |            0.159 |
| freshstack-laravel                   |        184 |    0.474 | pure_rrf        |      0.37  |      0.104 |           28.1 |            0.13  |            0.179 |
| freshstack-yolo                      |         57 |    0.364 | sparse_only     |      0.274 |      0.089 |           32.5 |            0.175 |            0.404 |
| gooaq                                |        119 |    0.617 | dense_only      |      0.601 |      0.016 |            2.7 |            0.101 |            0.16  |
| limit                                |        277 |    0.897 | sparse_only     |      0.894 |      0.003 |            0.4 |            0.487 |            0.065 |
| lotte-technology-forum               |       1261 |    0.566 | dense_only      |      0.488 |      0.078 |           15.9 |            0.125 |            0.106 |
| lotte-technology-search              |        395 |    0.816 | dense_only      |      0.755 |      0.061 |            8.1 |            0.073 |            0.061 |
| miracl-en-dev                        |        478 |    0.988 | dense_only      |      0.964 |      0.023 |            2.4 |            0.027 |            0     |
| msmarco-passage-dev                  |        491 |    0.938 | dense_only      |      0.823 |      0.115 |           14   |            0.595 |            0.012 |
| orcas                                |       1208 |    0.733 | dense_only      |      0.656 |      0.078 |           11.8 |            0.108 |            0.128 |
| quest                                |       1118 |    0.337 | pure_rrf        |      0.268 |      0.069 |           25.8 |            0.081 |            0.29  |
| rarb-code                            |        193 |    0.804 | dense_only      |      0.799 |      0.005 |            0.6 |            0.415 |            0.109 |
| rarb-math                            |        440 |    0.915 | dense_only      |      0.613 |      0.301 |           49.1 |            0.698 |            0.048 |
| scirgen-geo-en                       |       8030 |    0.289 | pure_rrf        |      0.245 |      0.044 |           17.8 |            0.048 |            0.54  |
| trec-dl-2022                         |         51 |    0.57  | dense_only      |      0.515 |      0.055 |           10.8 |            0.196 |            0.059 |
| webfaq-eng                           |       2065 |    0.87  | dense_only      |      0.839 |      0.032 |            3.8 |            0.08  |            0.051 |
| POOLED                               |      20815 |    0.494 | dense_only      |      0.404 |      0.089 |           22.1 |            0.125 |            0.311 |

| dataset                              |   dense_only |   pure_rrf |   sparse_only |
|:-------------------------------------|-------------:|-----------:|--------------:|
| antique                              |           10 |          0 |             6 |
| beir-nfcorpus                        |            6 |          0 |             5 |
| bright-biology                       |            0 |          2 |            11 |
| bright-earth-science                 |           11 |          0 |             7 |
| bright-economics                     |            3 |          0 |             1 |
| bright-leetcode                      |            2 |          0 |             1 |
| bright-pony                          |            0 |          1 |            16 |
| bright-psychology                    |            2 |          1 |             1 |
| bright-robotics                      |            4 |          2 |             3 |
| bright-stackoverflow                 |            3 |          2 |             6 |
| bright-sustainable-living            |            4 |          0 |             1 |
| bright-theoremqa-questions           |            5 |          0 |             0 |
| bright-theoremqa-theorems            |            0 |          0 |             2 |
| clerc                                |            5 |          1 |            30 |
| crumb-clinical-trial                 |            8 |          0 |             2 |
| crumb-code-retrieval                 |           49 |          0 |             9 |
| crumb-legal-qa                       |          412 |         10 |            18 |
| crumb-paper-retrieval                |            1 |          0 |             1 |
| crumb-set-operation-entity-retrieval |            8 |          0 |             8 |
| crumb-stack-exchange                 |            9 |          0 |             3 |
| crumb-tip-of-the-tongue              |            1 |          0 |             0 |
| dbpedia-entity                       |            6 |          0 |             6 |
| freshstack-angular                   |            7 |          2 |            11 |
| freshstack-godot                     |            8 |          2 |             1 |
| freshstack-langchain                 |           11 |          2 |            18 |
| freshstack-laravel                   |            6 |          1 |            17 |
| freshstack-yolo                      |            2 |          1 |             7 |
| gooaq                                |           11 |          1 |             0 |
| limit                                |            0 |          0 |           135 |
| lotte-technology-forum               |          109 |         16 |            32 |
| lotte-technology-search              |           24 |          2 |             3 |
| miracl-en-dev                        |           12 |          0 |             1 |
| msmarco-passage-dev                  |          252 |          1 |            39 |
| orcas                                |          101 |          5 |            24 |
| quest                                |           42 |          3 |            45 |
| rarb-code                            |           79 |          0 |             1 |
| rarb-math                            |          192 |          0 |           115 |
| scirgen-geo-en                       |          184 |         32 |           166 |
| trec-dl-2022                         |            8 |          0 |             2 |
| webfaq-eng                           |          141 |         10 |            15 |
| POOLED                               |         1738 |         97 |           769 |

ceiling over one global constant: +22.1% (0.404 -> 0.494)
lane-mix warning: scirgen-geo-en alone is 39% of labelled rows and its own ceiling is only +17.8%


## 19 — Route signal per cell (the prior, tested)

The point of the cell composition: does each archetype's a-priori `predicts`
route match what retrieval actually rewards? This joins the golden labels back
to `cell_selection` on `query_id` — the **exact** per-cell view, so a query
filling several cells counts under each — and sets the measured top route
(over the trainable `routes_differ` rows) against the cell's declared prior.

`prior_holds` is the falsification check: where measured ≠ predicted, either the
cell's prior is wrong or the lanes labelled so far are skewed. **Read it after
the sweep finishes** — mid-sweep the cheap, dense-friendly lanes dominate, so
almost everything reads `dense_only` until the technical/entity lanes land.

In [45]:
from composition.cells import CELLS

# exact per-cell view: re-join full cell membership so a query that fills
# several cells counts under each of them (the §7 markdown's promise)
cell_map = (
    selection[["dataset", "query_id", "cell"]]
    .astype({"query_id": str})
    .drop_duplicates()
)
done = labels.load()
done["query_id"] = done["query_id"].astype(str)
per_cell = done.drop(
    columns=[c for c in ("cell", "stage", "route_selected") if c in done]
).merge(cell_map, on=["dataset", "query_id"], how="left")

predicts = {cell.name: set(cell.predicts) for cell in CELLS}
rows = []
for cell, group in per_cell.groupby("cell"):
    differ = group[group["shape"] == "routes_differ"]
    dist = differ["route"].value_counts()
    top = dist.index[0] if len(dist) else None
    rows.append({
        "cell": cell or "(control)",
        "labelled": len(group),
        "trainable %": f"{len(differ) / len(group) * 100:.0f}%" if len(group) else "0%",
        "dense": int((differ["route"] == "dense_only").sum()),
        "rrf": int((differ["route"] == "pure_rrf").sum()),
        "sparse": int((differ["route"] == "sparse_only").sum()),
        "measured": top,
        "predicted": ",".join(sorted(predicts.get(cell, ()))) or "-",
        "prior_holds": (top in predicts.get(cell, set())) if top else None,
    })
per_cell_tbl = pd.DataFrame(rows).sort_values("labelled", ascending=False)
show(per_cell_tbl)

held = per_cell_tbl["prior_holds"].dropna()
print(f"predicted route is the measured top route in {int(held.sum())}/{len(held)} cells "
      f"— provisional: dense-friendly cheap lanes dominate until the sweep finishes")

| cell                             |   labelled | trainable %   |   dense |   rrf |   sparse | measured    | predicted              | prior_holds   |
|:---------------------------------|-----------:|:--------------|--------:|------:|---------:|:------------|:-----------------------|:--------------|
| stopword_saturated_midlength     |       4105 | 58%           |    1578 |   246 |      551 | dense_only  | dense_only,sparse_only | True          |
| multi_statement_context_dump     |       4025 | 57%           |    1403 |   141 |      758 | dense_only  | dense_only,pure_rrf    | True          |
| high_morphological_variation     |       2756 | 47%           |     664 |    93 |      529 | dense_only  | dense_only,sparse_only | True          |
| (control)                        |       2570 | 43%           |     612 |    44 |      452 | dense_only  | -                      | False         |
| wide_flat_enumeration            |       2094 | 55%           |     585 |   110 |      457 | dense_only  | pure_rrf,sparse_only   | False         |
| negation_bearing_question        |       1766 | 48%           |     552 |    72 |      219 | dense_only  | dense_only,sparse_only | True          |
| extreme_length_pasted_query      |       1710 | 73%           |     587 |   156 |      502 | dense_only  | dense_only,sparse_only | True          |
| comparative_multi_entity         |       1075 | 56%           |     326 |    47 |      224 | dense_only  | dense_only,pure_rrf    | True          |
| relative_temporal_no_dates       |        893 | 57%           |     336 |    46 |      127 | dense_only  | dense_only             | True          |
| short_grammatical_question       |        745 | 66%           |     296 |    68 |      131 | dense_only  | dense_only,sparse_only | True          |
| verbose_grammatical_request      |        669 | 64%           |     288 |    17 |      122 | dense_only  | dense_only,pure_rrf    | True          |
| number_inside_natural_question   |        549 | 54%           |     140 |    37 |      118 | dense_only  | dense_only,sparse_only | True          |
| deep_nesting_single_sentence     |        488 | 57%           |     137 |    38 |      102 | dense_only  | dense_only             | True          |
| keyword_telegram_short           |        339 | 77%           |     141 |    40 |       79 | dense_only  | dense_only,sparse_only | True          |
| conversational_courtesy_wrapper  |        313 | 48%           |      88 |    13 |       50 | dense_only  | dense_only             | True          |
| bare_concept_token               |        257 | 68%           |      99 |    31 |       46 | dense_only  | dense_only             | True          |
| acronym_inside_question          |        236 | 65%           |      92 |    24 |       38 | dense_only  | pure_rrf,sparse_only   | False         |
| capsword_shape_ambiguity         |        212 | 70%           |      96 |    30 |       22 | dense_only  | dense_only,sparse_only | True          |
| version_pinned_technical         |        177 | 56%           |      68 |     2 |       30 | dense_only  | pure_rrf,sparse_only   | False         |
| package_coordinate_dependency    |        139 | 83%           |      41 |    27 |       48 | sparse_only | pure_rrf,sparse_only   | True          |
| code_symbol_named_in_prose       |        137 | 45%           |      53 |     2 |        7 | dense_only  | pure_rrf,sparse_only   | False         |
| instance_value_in_intent         |        133 | 81%           |      63 |     2 |       43 | dense_only  | dense_only             | True          |
| registry_structured_identifier   |        104 | 40%           |      32 |     1 |        9 | dense_only  | sparse_only            | False         |
| web_locator_token                |         95 | 59%           |      48 |     0 |        8 | dense_only  | dense_only,sparse_only | True          |
| status_code_idf_split            |         66 | 45%           |      16 |     2 |       12 | dense_only  | dense_only,sparse_only | True          |
| math_notation_present            |         62 | 87%           |      48 |     0 |        6 | dense_only  | dense_only,sparse_only | True          |
| short_quantified_spec            |         58 | 71%           |      32 |     5 |        4 | dense_only  | dense_only,sparse_only | True          |
| geo_coordinate_postal            |         44 | 43%           |       8 |     0 |       11 | sparse_only | dense_only,sparse_only | True          |
| env_var_configuration            |         39 | 74%           |      12 |     4 |       13 | sparse_only | pure_rrf,sparse_only   | True          |
| pasted_code_fragment             |         33 | 85%           |      14 |     4 |       10 | dense_only  | pure_rrf,sparse_only   | False         |
| bare_machine_token               |         17 | 53%           |       8 |     0 |        1 | dense_only  | sparse_only            | False         |
| bare_acronym                     |         15 | 47%           |       2 |     3 |        2 | pure_rrf    | dense_only,sparse_only | False         |
| rare_key_buried_in_chatter       |         14 | 57%           |       2 |     0 |        6 | sparse_only | pure_rrf,sparse_only   | True          |
| bare_number_token                |         14 | 50%           |       3 |     0 |        4 | sparse_only | dense_only,sparse_only | True          |
| boolean_operator_query           |          6 | 67%           |       2 |     0 |        2 | sparse_only | dense_only,sparse_only | True          |
| standards_compliance_lookup      |          2 | 100%          |       2 |     0 |        0 | dense_only  | pure_rrf,sparse_only   | False         |
| bibliographic_catalog_identifier |          2 | 100%          |       2 |     0 |        0 | dense_only  | dense_only,sparse_only | True          |
| logistics_catalog_token          |          2 | 100%          |       2 |     0 |        0 | dense_only  | sparse_only            | False         |
| datetime_token_present           |          1 | 0%            |       0 |     0 |        0 | nan         | dense_only,sparse_only |               |

predicted route is the measured top route in 27/38 cells — provisional: dense-friendly cheap lanes dominate until the sweep finishes
